In [28]:
import polars as pl
import pandas as pd
import json

In [29]:
DATA_FILES = {
    "emission_forecast": "datasafehouse-emission-forecast-export_all_v3_20260709live.csv",
    "demand_forecast": "datasafehouse-forecast-export_all_v3_20260709live.csv",
    "emission_reference": "reference_emission_data_20260709live.csv",
    "demand_reference": "reference_utility_data_20260709live.csv",
    "plants": "datasafehouse-plant-export_20260709live.csv",
    "project_emissions": "datasafehouse-projectdata-export_emissions_20260709live.csv",
    "project_utilities": "datasafehouse-projectdata-export_utilities_20260709live.csv",
    "production": "data-export_electricity_production_20260709live.csv",
    "storage": "data-export_energy_storage_20260709live.csv",
    "flexibility": "data-export_flex_options_20260709live.csv",
}

DATA_DIR = '/home/307920@ontw.alfa.local/projects/epn-ma-master/data/dsh/20260709_live DSH exports'

def parse_All_EANs(df: pl.DataFrame) -> pl.DataFrame:
    """Expands the 'All EANs' JSON column into multiple rows/columns."""
    df_pd = df.to_pandas()
    df_pd['All EANs'] = df_pd['All EANs'].apply(json.loads)
    df_pd = df_pd.explode('All EANs', ignore_index=True)
    ean_df = pd.json_normalize(df_pd['All EANs'])
    df_pd = pd.concat([df_pd.drop('All EANs', axis=1), ean_df], axis=1)
    return pl.from_pandas(df_pd)

def read_csv(filename: str) -> pl.DataFrame:
    """Read a CSV via pandas, falling back to Polars for malformed files."""
    filepath = f"{DATA_DIR}/{filename}"
    try:
        return pl.from_pandas(pd.read_csv(filepath))
    except pd.errors.ParserError:
        print(f"  [warn] Falling back to Polars for {filename}")
        return pl.read_csv(filepath, truncate_ragged_lines=True)
 
 
def load_data() -> dict[str, pl.DataFrame]:
    """Load all source CSVs and return as a named dict."""
    print("Loading data...")
    data = {name: read_csv(filename) for name, filename in DATA_FILES.items()}
    # Parse EANs once here so it's not repeated per plant
    data["plants_parsed"] = parse_All_EANs(data["plants"])
    print(f"  Loaded {len(DATA_FILES)} files.")
    return data
 

In [30]:
data = load_data()

Loading data...
  [warn] Falling back to Polars for data-export_electricity_production_20260709live.csv
  [warn] Falling back to Polars for data-export_flex_options_20260709live.csv
  Loaded 10 files.


In [31]:
YEARS = ['2024', '2030', '2035', '2040', '2050']
SCENARIO_YEARS = ['2030', '2035', '2040', '2050']
REF_YEAR = '2024'

In [32]:
data['plants'] = data['plants'].select(['Plant name', 'Plant identifier'])

In [33]:
data['emission_reference'] = (
    data['emission_reference'].select(
    ['Plant', 'Plant identifier', 'Cluster','Emission', 'Year', 'Annual amount'])
    .filter(pl.col('Year').cast(pl.Utf8) == REF_YEAR)
    .rename({'Annual amount' : 'Value'})
    .pivot('Emission', index=['Plant identifier', 'Plant', 'Cluster', 'Year'], values='Value')
    .with_columns(pl.col('Year').cast(pl.Utf8))
    .with_columns(pl.lit("Reference").alias("Scenario"))
    .with_columns(pl.lit('production').alias('Flow type'))
    .with_columns(
                (pl.col("NOx").cast(pl.Float64).fill_null(0.0) 
                 + pl.col("N2O").cast(pl.Float64).fill_null(0.0)
                 ).alias("N2O")
            )
    .drop(['NOx'])
    .rename({'Plant':'Plant name'})
)

In [34]:
data['emission_forecast'] = (
    data['emission_forecast'].drop(['Company', 'Version date'])
    .unpivot(SCENARIO_YEARS, index = ['Plant name', 'Plant identifier', 'Scenario', 'Emission'], variable_name='Year', value_name='Value')
    .pivot('Emission', index=['Plant identifier', 'Plant name', 'Scenario', 'Year'], values='Value')
    .with_columns(pl.lit('production').alias('Flow type'))
    .with_columns(
                    (pl.col("NOx").cast(pl.Float64).fill_null(0.0) 
                        + pl.col("N₂O").cast(pl.Float64).fill_null(0.0)
                        ).alias("N2O")
                )
        .drop(['NOx', 'N₂O'])
        )

In [35]:
data['emission_reference'].schema

Schema([('Plant identifier', String),
        ('Plant name', String),
        ('Cluster', String),
        ('Year', String),
        ('CO2', Float64),
        ('N2O', Float64),
        ('Scenario', String),
        ('Flow type', String)])

In [36]:
data['emission_forecast'].schema

Schema([('Plant identifier', String),
        ('Plant name', String),
        ('Scenario', String),
        ('Year', String),
        ('CO2', Float64),
        ('Methane', Float64),
        ('F-gases', Float64),
        ('other', Float64),
        ('Flow type', String),
        ('N2O', Float64)])

In [37]:
result = pl.concat(
    [data['emission_reference'], data['emission_forecast']],
    how="diagonal_relaxed",
)

result

Plant identifier,Plant name,Cluster,Year,CO2,N2O,Scenario,Flow type,Methane,F-gases,other
str,str,str,str,f64,f64,str,str,f64,f64,f64
"""527be9a2-5258-4cda-89cb-b48520…","""ADM Europoort""","""Rotterdam-Moerdijk""","""2024""",135.297,0.0,"""Reference""","""production""",null,null,null
"""18ada8d6-9982-4647-a80b-90fb14…","""Alco Energy Europoort""","""Rotterdam-Moerdijk""","""2024""",348.0,0.0,"""Reference""","""production""",null,null,null
"""cb69c2cf-c121-46d5-84a3-35ea3b…","""ExxonMobil Raffinaderij Botlek""","""Rotterdam-Moerdijk""","""2024""",2182.0,0.0,"""Reference""","""production""",null,null,null
"""3607cefd-8e6c-4943-8878-2e2cc7…","""Lyondell Botlek""","""Rotterdam-Moerdijk""","""2024""",290.653,0.0,"""Reference""","""production""",null,null,null
"""c00446f2-df0c-466d-a5c9-82f434…","""Nobian Botlek""","""Rotterdam-Moerdijk""","""2024""",103.0,0.0,"""Reference""","""production""",null,null,null
…,…,…,…,…,…,…,…,…,…,…
"""9f81eb85-64e2-44ed-8f85-beb6a9…","""Yara Sluiskil""",null,"""2050""",2182.5,6.0,"""CCS and (green) gas""","""production""",null,null,null
"""401a65fd-5228-47d2-ae6a-9622a8…","""Zeeland Refinery""",null,"""2050""",816.0,0.0,"""Preferred""","""production""",null,null,null
"""401a65fd-5228-47d2-ae6a-9622a8…","""Zeeland Refinery""",null,"""2050""",200.0,0.0,"""Electrification""","""production""",null,null,null


In [38]:
type(data)

dict

In [39]:
"""
Build master wide-format table from DSH data files.
"""

import polars as pl
from typing import Dict


def build_master_table(data: Dict[str, pl.DataFrame]) -> pl.DataFrame:
    """
    Build master table from DSH forecasts and reference data.
    
    Wide format:
    Company | Cluster | Sector | Scenario | Year | Type | CO2 | Methane | ... | Electricity | Natural Gas | ...
    
    Args:
        data: Dict with keys: emission_forecast, demand_forecast, 
              emission_reference, demand_reference, plants
    
    Returns:
        Master DataFrame in wide format
    """
    
    # ── Step 1: Get cluster/sector from reference data ─────────────────
    # Get unique company-cluster-plant pairs
    ref_meta = data["demand_reference"].select([
        "Plant identifier", "Company", "Cluster"
    ]).unique()
    
    # Rename SBI to Sector (or create from SBI code if needed)
    # ref_meta = ref_meta.rename({"SBI (2008)": "Sector"})
    
    # ── Step 2: Reshape emission forecast ──────────────────────────────
    # Pivot years to long format, then pivot Emission to wide
    emission_long = data["emission_forecast"].unpivot(
        index=["Plant identifier", "Company", "Scenario", "Emission"],
        variable_name="Year",
        value_name="Value"
    )
    
    # Convert Year to int
    # emission_long = emission_long.with_columns(
    #     pl.col("Year").cast(pl.Utf8).str.to_integer().cast(pl.Int64)
    # )
    
    # Pivot Emission types to columns (CO2, Methane, etc)
    emission_wide = emission_long.pivot(
        index=["Plant identifier", "Company", "Scenario", "Year"],
        columns="Emission",
        values="Value",
        aggregate_function="first"
    )
    
    # ── Step 3: Reshape demand forecast ────────────────────────────────
    # Pivot years to long format
    demand_long = data["demand_forecast"].unpivot(
        index=["Plant identifier", "Company", "Scenario", "Utility", "Flow type"],
        variable_name="Year",
        value_name="Value"
    )
    
    # Convert Year to int
    # demand_long = demand_long.with_columns(
    #     pl.col("Year").cast(pl.Utf8).str.to_integer().cast(pl.Int64)
    # )
    
    # Rename "Flow type" to "Type"
    demand_long = demand_long.rename({"Flow type": "Type"})
    
    # Pivot Utilities to columns (Electricity, Natural Gas, etc)
    demand_wide = demand_long.pivot(
        index=["Plant identifier", "Company", "Scenario", "Year", "Type"],
        columns="Utility",
        values="Value",
        aggregate_function="first"
    )
    
    # ── Step 4: Join emissions + demands ───────────────────────────────
    # Merge on Plant identifier, Company, Scenario, Year
    master = emission_wide.join(
        demand_wide,
        on=["Plant identifier", "Company", "Scenario", "Year"],
        how="outer"
    ).drop("Company_right")
    
    # ── Step 5: Add cluster/sector metadata ────────────────────────────
    master = master.join(
        ref_meta,
        on="Plant identifier",
        how="left"
    )
    
    # Handle duplicate Company columns (keep the one from ref_meta)
    if master.columns.count("Company") > 1:
        master = master.drop("Company_right").rename({"Company_left": "Company"})
    
    # ── Step 6: Reorder columns ────────────────────────────────────────
    # Priority: Company, Cluster, Sector, Scenario, Year, Type, then data columns
    base_cols = ["Company", "Cluster", "Sector", "Scenario", "Year", "Type"]
    data_cols = [c for c in master.columns if c not in base_cols + ["Plant identifier"]]
    
    final_cols = base_cols + data_cols + ["Plant identifier"]
    
    # Only keep columns that exist
    final_cols = [c for c in final_cols if c in master.columns]
    
    master = master.select(final_cols)
    
    # ── Step 7: Sort for readability ───────────────────────────────────
    master = master.sort(["Company", "Scenario", "Year", "Type"])
    
    return master




In [40]:
# master = build_master_table(data)
# master.write_csv("master_table.csv")
# st.dataframe(master)

In [41]:
# master

In [42]:
from read_DSH_files import read_all_scenario_sheets
from push_to_ctm_modules import load_all_plants_scenario_data
from constants import EMISSION_COLS_ORDER, UTILITY_COLS_ORDER

test_path = '/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/Sprint 2 CTM upload/Cargill Bergen op Zoom.xlsx'

test_plant = test_path.split('/')[-1].split('.')[0]

test = read_all_scenario_sheets(
    workbook_path=test_path,
    emission_cols=EMISSION_COLS_ORDER,
    energy_cols=UTILITY_COLS_ORDER,
    reference_year=REF_YEAR,
    aggregate_flow_types=False
)



In [43]:
aux_details = data['emission_reference'].filter(pl.col('Plant name') == test_plant)
aux_details

Plant identifier,Plant name,Cluster,Year,CO2,N2O,Scenario,Flow type
str,str,str,str,f64,f64,str,str
"""8848af70-221e-4489-b24e-565e00…","""Cargill Bergen op Zoom""","""Zeeland-West-Brabant""","""2024""",44.5,0.0,"""Reference""","""production"""


In [44]:
cluster = aux_details.select(pl.col('Cluster'))[0].item()
cluster

'Zeeland-West-Brabant'

In [45]:
aux_ref = data['emission_reference'].select(['Scenario', 'Year', 'Flow type', 'CO2', 'N2O'])
aux_ref

Scenario,Year,Flow type,CO2,N2O
str,str,str,f64,f64
"""Reference""","""2024""","""production""",135.297,0.0
"""Reference""","""2024""","""production""",348.0,0.0
"""Reference""","""2024""","""production""",2182.0,0.0
"""Reference""","""2024""","""production""",290.653,0.0
"""Reference""","""2024""","""production""",103.0,0.0
…,…,…,…,…
"""Reference""","""2024""","""production""",147.0,0.0
"""Reference""","""2024""","""production""",null,0.0
"""Reference""","""2024""","""production""",219.835,0.098282


In [46]:
def process_demand_reference(df: pl.DataFrame) -> pl.DataFrame:
    """
    Process demand_reference to match scenario data schema.
    
    Converts:
    - Year: Int64 → String
    - Creates Flow type column from Annual demand/supply/production/captive use
    - Pivots utilities to columns
    
    Returns same schema as scenario sheets:
    Plant identifier, Plant name, Cluster, Year, <utilities>, Scenario, Flow type
    """
    
    # Convert Year to String
    df = df.with_columns(
        pl.col("Year").cast(pl.Utf8)
    )
    
    # Create long format with different flow types
    demand_rows = df.select([
        "Plant identifier", "Plant", "Cluster", "Year", "Utility",
        pl.lit("demand").alias("Flow type"),
        pl.lit("Reference").alias("Scenario"),
        pl.col("Annual demand").alias("Value"),
    ]).filter(pl.col("Value").is_not_null())
    
    supply_rows = df.select([
        "Plant identifier", "Plant", "Cluster", "Year", "Utility",
        pl.lit("supply").alias("Flow type"),
        pl.lit("Reference").alias("Scenario"),
        pl.col("Annual supply").alias("Value"),
    ]).filter(pl.col("Value").is_not_null())
    
    production_rows = df.select([
        "Plant identifier", "Plant", "Cluster", "Year", "Utility",
        pl.lit("production").alias("Flow type"),
        pl.lit("Reference").alias("Scenario"),
        pl.col("Annual production").alias("Value"),
    ]).filter(pl.col("Value").is_not_null())
    
    captive_rows = df.select([
        "Plant identifier", "Plant", "Cluster", "Year", "Utility",
        pl.lit("captive use").alias("Flow type"),
        pl.lit("Reference").alias("Scenario"),
        pl.col("Captive use annual").alias("Value"),
    ]).filter(pl.col("Value").is_not_null())
    
    # Combine all flow types
    combined = pl.concat([demand_rows, supply_rows, production_rows, captive_rows])
    
    # Pivot utilities to columns (like scenario data)
    pivoted = combined.pivot(
        index=["Plant identifier", "Plant", "Cluster", "Year", "Scenario", "Flow type"],
        columns="Utility",
        values="Value",
        aggregate_function="first"
    )
    
    # Rename "Plant" to "Plant name" for consistency
    pivoted = pivoted.rename({"Plant": "Plant name"})
    
    # Reorder columns: Plant identifier, Plant name, Cluster, Year, <utilities>, Scenario, Flow type
    base_cols = ["Plant identifier", "Plant name", "Cluster", "Year"]
    utility_cols = [c for c in pivoted.columns if c not in base_cols + ["Scenario", "Flow type"]]
    final_cols = base_cols + utility_cols + ["Scenario", "Flow type"]
    
    return pivoted.select(final_cols)

In [47]:
data["demand_reference_processed"] = process_demand_reference(data["demand_reference"])


/tmp/ipykernel_925441/1109463735.py:52: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  pivoted = combined.pivot(


In [48]:
data["demand_reference_processed"]

Plant identifier,Plant name,Cluster,Year,Electricity,Natural Gas,Heat,Oil and oil products,Coal and coal products,Hydrogen ( >98% vol.%) (LHV),Other syn fuel and raw materials,Hydrogen ( <98% vol.%) (LHV),Residual gases,Waste (fossil),Waste (bio),Biomass (liquid),CO2 (bio) CCU/CCS,CO2 (fossil) CCU/CCS,Scenario,Flow type
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""6beec4ef-263a-418f-a992-34ad7a…","""Chane Terminal Zaandam (CTZA)""","""Noordzeekanaalgebied""","""2024""",0.311,8.7,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""demand"""
"""ec1ffca3-e31f-4b8c-b466-681151…","""Chane Terminal Pernis (CTPE)""","""Rotterdam-Moerdijk""","""2022""",7.16,71.994,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""demand"""
"""ec1ffca3-e31f-4b8c-b466-681151…","""Chane Terminal Pernis (CTPE)""","""Rotterdam-Moerdijk""","""2024""",6.67,29.0,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""demand"""
"""ec1ffca3-e31f-4b8c-b466-681151…","""Chane Terminal Pernis (CTPE)""","""Rotterdam-Moerdijk""","""2021""",6.858,75.735,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""demand"""
"""277ed4ca-ce15-4bd9-b026-b322ed…","""AsfaltNU ANA II""","""Noordzeekanaalgebied""","""2023""",1.64,12.3,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""demand"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2ea6230e-8e15-4765-b40e-b2d20c…","""Ketjen CMF""","""Noordzeekanaalgebied""","""2021""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""captive use"""
"""2ea6230e-8e15-4765-b40e-b2d20c…","""Ketjen CMF""","""Noordzeekanaalgebied""","""2024""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""captive use"""
"""42aec4bc-d7a6-44b1-8427-cf8360…","""EEW Delfzijl""","""Noord-Nederland""","""2024""",42.9,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Reference""","""captive use"""


In [49]:

from openpyxl import load_workbook

def read_all_scenario_sheets_from_excels(
    excel_files_dict: Dict[str, bytes],
    emission_cols: list[str],
    energy_cols: list[str],
    reference_year: int = 2024,
) -> pl.DataFrame:
    """
    Read all Scenario X sheets from all generated Excel files.
    Returns combined DataFrame with plant name from filename.
    """
    
    import tempfile
    from pathlib import Path
    
    all_records = []
    
    for file_name, file_bytes in excel_files_dict.items():
        # Extract plant name from filename (e.g., "Plant A.xlsx" → "Plant A")
        plant_name = Path(file_name).stem
        
        # Save to temp file
        with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as tmp:
            tmp.write(file_bytes)
            tmp_path = tmp.name
        
        try:
            wb = load_workbook(tmp_path, data_only=True)
            
            # Read each Scenario sheet
            for sheet_name in wb.sheetnames:
                if not sheet_name.startswith("Scenario"):
                    continue
                
                df = read_all_scenario_sheets(
                    workbook_path=tmp_path,
                    sheet_name=sheet_name,
                    emission_cols=emission_cols,
                    energy_cols=energy_cols,
                    reference_year=reference_year,
                )
                
                if not df.is_empty():
                    # Add plant name column
                    df = df.with_columns(
                        pl.lit(plant_name).alias("Plant name")
                    )
                    all_records.append(df)
        
        finally:
            Path(tmp_path).unlink()  # Clean up temp file
    
    if all_records:
        return pl.concat(all_records)
    else:
        return pl.DataFrame()

In [50]:
def build_master_table_combined(
    scenario_data: pl.DataFrame,  # From Excel files (has Scenario, Year, Flow type, utilities, emissions)
    emission_reference: pl.DataFrame,  # Already processed
    demand_reference: pl.DataFrame,  # Already processed (processed_demand_reference)
    plant_meta: pl.DataFrame,  # For enriching scenario data with Plant identifier, Cluster
) -> pl.DataFrame:
    """
    Combine scenario data, emission reference, and demand reference into one master table.
    
    Args:
        scenario_data: DataFrame from reading Excel scenario sheets
        emission_reference: Processed emission reference data
        demand_reference: Processed demand reference data (from process_demand_reference)
        plant_meta: DataFrame with Plant name, Plant identifier, Cluster mapping
    
    Returns:
        Master table with all data combined
    """
    
    # ── Part 1: Enrich scenario data with plant metadata ────────────────
    # Scenario data needs: Plant identifier, Plant name, Cluster
    # (assuming it comes from a specific Excel file, so we know the plant)
    
    scenario_enriched = scenario_data.join(
        plant_meta.select(["Plant name", "Plant identifier", "Cluster"]),
        left_on="Plant name",  # or however plant is identified in Excel
        right_on="Plant name",
        how="left"
    )
    
    # Reorder scenario columns to match reference data
    base_cols = ["Plant identifier", "Plant name", "Cluster", "Year", "Scenario", "Flow type"]
    data_cols = [c for c in scenario_enriched.columns if c not in base_cols]
    scenario_enriched = scenario_enriched.select(base_cols + data_cols)
    
    # ── Part 2: Standardize all three dataframes ───────────────────────
    # Find all unique columns across all three
    all_cols = set(scenario_enriched.columns) | set(emission_reference.columns) | set(demand_reference.columns)
    
    # Common base columns
    base_cols = ["Plant identifier", "Plant name", "Cluster", "Year", "Scenario", "Flow type"]
    
    # Data columns (metrics)
    metric_cols = sorted([c for c in all_cols if c not in base_cols])
    
    # Fill missing columns with null
    for df in [scenario_enriched, emission_reference, demand_reference]:
        for col in metric_cols:
            if col not in df.columns:
                df = df.with_columns(pl.lit(None).cast(pl.Float64).alias(col))
    
    # ── Part 3: Concatenate all three ──────────────────────────────────
    master = pl.concat(
        [scenario_enriched, emission_reference, demand_reference],
        how="diagonal_relaxed"  # Handles different column sets
    )
    
    # ── Part 4: Reorder and sort ───────────────────────────────────────
    final_cols = base_cols + metric_cols
    final_cols = [c for c in final_cols if c in master.columns]
    
    master = master.select(final_cols)
    
    # Sort for readability
    master = master.sort_by(["Plant name", "Scenario", "Year", "Flow type"])
    
    return master

In [51]:
EXCEL_FOLDER = '/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/Sprint 2 CTM upload'
from pathlib import Path

excel_folder = Path(EXCEL_FOLDER)
excel_files = list(excel_folder.glob("*.xlsx"))
 
print(f"Found {len(excel_files)} Excel files")
 
all_scenario_records = []
 
for excel_path in excel_files:  # Test with first 5 files
    plant_name = excel_path.stem
    print(f"  Reading: {plant_name}")
    
    try:
        wb = load_workbook(excel_path, data_only=True)
        
           
        df = read_all_scenario_sheets(
            workbook_path=str(excel_path),
            emission_cols=EMISSION_COLS_ORDER,
            energy_cols=UTILITY_COLS_ORDER,
            reference_year='0',
            aggregate_flow_types=False, 
        )

        reference = df.filter(
            pl.col("Year") == str(REF_YEAR)
        )
        
        # Set Scenario to 'Reference'
        reference = reference.with_columns(
            pl.lit("Reference").alias("Scenario")
        )
        
        # Remove duplicates (same plant, year, flow_type should have same metric values)
        reference = reference.unique(
            subset=["Scenario", "Year", "Flow type"],
            keep="first"
        )

        scenario_data = df.filter(pl.col("Year") != "2024")

        # Combine
        master = pl.concat([scenario_data, reference])

        if not master.is_empty():
            # Add plant name
            master = master.with_columns(
                pl.lit(plant_name).alias("Plant name")
            )
            all_scenario_records.append(master)
            
    
    except Exception as e:
        print(f"    [ERROR] {e}")
 
if all_scenario_records:
    scenario_data = pl.concat(all_scenario_records, how='vertical_relaxed')
    print(f"Combined scenario data: {scenario_data.shape}")
    print(f"Columns: {scenario_data.columns}")
else:
    print("No scenario data found!")
    scenario_data = pl.DataFrame()

Found 46 Excel files
  Reading: Air Liquide Bergen op Zoom
  Reading: Air Liquide Booster Moerdijk
  Reading: Air Liquide Moerdijk
  Reading: ADM Europoort
  Reading: Air Liquide Pernis
  Reading: Air Liquide Rozenburg (Botlek)
  Reading: Air Liquide Terneuzen
  Reading: Air Products locatie Botlekweg (HyCO)
  Reading: Air Products locatie Merseyweg (ASUs)
  Reading: Alco Energy Europoort
  Reading: AsfaltNU ANA I
  Reading: Cargill Aurora
  Reading: AsfaltNU ANA II
  Reading: Cabot Botlek
  Reading: Cargill Botlek
  Reading: Cargill Jonker
  Reading: Cargill Bergen op Zoom
  Reading: Cargill Multiseeds
  Reading: Cargill ZOR
  Reading: Delfzijl Nieuw
  Reading: CiC Chemport Innovation Center
  Reading: Dow Delfzijl
  Reading: ExxonMobil Raffinaderij Botlek
  Reading: ExxonMobil Europoort
  Reading: Forbo Flooring BV
  Reading: Huntsman Botlek
  Reading: Getec Park Emmen
  Reading: Ketjen CMF
  Reading: Lanxess Botlek
  Reading: LW Bergen op Zoom
  Reading: LW Kruiningen
  Reading: Lyo

In [52]:
scenario_data

Scenario,Year,Flow type,CO2,Methane,N2O,F-gases,CO2 (fossil) CCU/CCS,CO2 (bio) CCU/CCS,Electricity,Electricity_peak,Natural Gas,Hydrogen ( >98% vol.%) (LHV),Hydrogen ( <98% vol.%) (LHV),Heat,Residual gases,Coal and coal products,Oil and oil products,Biomass (liquid),Biomass (solid),Green gas,Waste (fossil),Waste (bio),Ammonia,Methanol,Other syn fuel and raw materials,Other,Plant name
str,str,str,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,i64,i64,f64,f64,i64,str
"""Midden""","""2030""","""demand""",null,null,null,null,null,null,39.0,7.0,627.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Air Liquide Bergen op Zoom"""
"""Midden""","""2030""","""captive use""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Air Liquide Bergen op Zoom"""
"""Midden""","""2030""","""production""",null,null,null,null,null,null,null,null,null,null,null,67.9,null,null,null,null,null,null,null,null,null,null,null,null,"""Air Liquide Bergen op Zoom"""
"""Midden""","""2030""","""supply""",null,null,null,null,null,null,null,null,null,-157.0,null,67.9,null,null,null,null,null,null,null,null,null,null,null,null,"""Air Liquide Bergen op Zoom"""
"""Midden""","""2035""","""demand""",null,null,null,null,null,null,39.0,7.0,627.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Air Liquide Bergen op Zoom"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VT""","""2050""","""supply""",0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.3,0.0,0,0.0,0,0.0,0.0,0,0,0,0.0,0.0,null,"""Zeeland Refinery"""
"""Reference""","""2024""","""production""",1528.0,null,null,null,null,null,10.6,null,null,3673.0,null,null,5018.0,null,null,null,null,null,null,null,null,null,null,null,"""Zeeland Refinery"""
"""Reference""","""2024""","""demand""",null,null,null,null,null,null,368.2,48.0,2369.0,0.0,null,null,null,null,110837.6,null,null,null,null,null,null,null,null,null,"""Zeeland Refinery"""


In [53]:
# mapping = pl.read_csv('/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/mapping.csv')
from utils import read_and_transform_mapping

map_path = '/home/307920@ontw.alfa.local/projects/epn-ma-master/data/ctm/input/20260630 sprint 2 CTM-DSH site mapping.xlsx'

mapping = read_and_transform_mapping(
    excel_path=map_path, 
    save_file=False,
    normalize_sector_cluster=True

)


Could not determine dtype for column 4, falling back to string


Session created: SE-a36c49848deb095c
Clean sheet applied.
Loading all arguments...
Session deleted: SE-a36c49848deb095c successfully deleted!


In [54]:
mapping_subset = mapping.select([
        "DSH plant name",
        "Cluster",
        "Sector",
    ]).unique()
    
# Join on plant name
df_enriched = scenario_data.join(
    mapping_subset,
    left_on='Plant name',
    right_on="DSH plant name",
    how="left"
)

# Drop DSH plant name (duplicate)
if "DSH plant name" in df_enriched.columns:
    df_enriched = df_enriched.drop("DSH plant name")

# Reorder: Cluster and Sector after plant info, before metrics
base_cols = ["Plant name", "Cluster", "Sector", "Scenario", "Year", "Flow type"]
metric_cols = [c for c in df_enriched.columns if c not in base_cols]

df_enriched = df_enriched.select(base_cols + metric_cols)

In [55]:
df_enriched.sort(['Plant name', 'Year', 'Flow type'])

Plant name,Cluster,Sector,Scenario,Year,Flow type,CO2,Methane,N2O,F-gases,CO2 (fossil) CCU/CCS,CO2 (bio) CCU/CCS,Electricity,Electricity_peak,Natural Gas,Hydrogen ( >98% vol.%) (LHV),Hydrogen ( <98% vol.%) (LHV),Heat,Residual gases,Coal and coal products,Oil and oil products,Biomass (liquid),Biomass (solid),Green gas,Waste (fossil),Waste (bio),Ammonia,Methanol,Other syn fuel and raw materials,Other
str,str,str,str,str,str,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,i64,i64,f64,f64,i64
"""ADM Europoort""","""rotterdam_moerdijk""","""food""","""Reference""","""2024""","""captive use""",null,null,null,null,null,null,121.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ADM Europoort""","""rotterdam_moerdijk""","""food""","""Reference""","""2024""","""demand""",null,null,null,null,null,null,2.16,10.0,743.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ADM Europoort""","""rotterdam_moerdijk""","""food""","""Reference""","""2024""","""production""",135.297,null,null,null,null,null,160.329,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ADM Europoort""","""rotterdam_moerdijk""","""food""","""Reference""","""2024""","""supply""",null,null,null,null,null,null,41.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ADM Europoort""","""rotterdam_moerdijk""","""food""","""Groen gas""","""2030""","""captive use""",0.0,0.0,0,0.0,0.0,0.0,121.0,17.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0,0,0,0.0,0.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zeeland Refinery""","""zeeland_west_brabant""","""refineries""","""Midden""","""2050""","""supply""",0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.3,0.0,0,0.0,0,0.0,0.0,0,0,0,0.0,0.0,null
"""Zeeland Refinery""","""zeeland_west_brabant""","""refineries""","""Groen gas""","""2050""","""supply""",0.0,0.0,0,0.0,0.0,656.0,0.0,0.0,0.0,0.0,0.0,6.3,0.0,0,0.0,0,0.0,0.0,0,0,0,0.0,0.0,null
"""Zeeland Refinery""","""zeeland_west_brabant""","""refineries""","""Waterstof""","""2050""","""supply""",0.0,0.0,0,0.0,0.0,656.0,0.0,0.0,0.0,0.0,0.0,6.3,0.0,0,0.0,0,0.0,0.0,0,0,0,0.0,0.0,null


In [56]:
df_enriched.write_excel('test_master.xlsx')

In [57]:

clusters = mapping.select(pl.col('Sector')).unique()
clusters.to_series().to_list()

['other_metals',
 'refineries',
 'inorganic_base_chemicals',
 'central_ict',
 'other',
 'textile_and_leather',
 'mining_and_quarrying',
 'Construction',
 'machinery',
 'organic_base_chemicals',
 'Wood and wood products',
 'aluminium',
 'paper',
 'food',
 'other_chemicals',
 'transport_equipment',
 'non_metallic_minerals',
 'gebruiker_stuurt_op_via_api',
 'steel']

In [58]:
cluster_map = {
    'nzkg': 'nzkg',
    'NZKG': 'nzkg',
    'rotterdam_moerdijk': 'rotterdam_moerdijk',
    'Rotterdam-Moerdijk': 'rotterdam_moerdijk',
    'rotterdam moerdijk': 'rotterdam_moerdijk',
    'cluster_6': 'cluster_6',
    'Cluster 6': 'cluster_6',
    'cluster 6': 'cluster_6',
    'noord_nederland': 'noord_nederland',
    'Noord-Nederland': 'noord_nederland',
    'noord nederland': 'noord_nederland',
    'zeeland_west_brabant': 'zeeland_west_brabant',
    'Zeeland-West-Brabant': 'zeeland_west_brabant',
    'zeeland west brabant': 'zeeland_west_brabant',
    'Gebruiker stuurt op via API': 'gebruiker_stuurt_op_via_api',
}

# Sector normalizations
sector_map = {
    'Organic base': 'organic_base_chemicals',
    'Organic base chemicals': 'organic_base_chemicals',
    'organic_base_chemicals': 'organic_base_chemicals',
    'Inorganic base chemicals': 'inorganic_base_chemicals',
    'inorganic_base_chemicals': 'inorganic_base_chemicals',
    'Other chemicals': 'other_chemicals',
    'other_chemicals': 'other_chemicals',
    'Other chemical': 'other_chemicals',
    'Food': 'food',
    'food': 'food',
    'Steel': 'steel',
    'steel': 'steel',
    'Refineries': 'refineries',
    'refineries': 'refineries',
    'Aluminium': 'aluminium',
    'aluminium': 'aluminium',
    'Paper': 'paper',
    'paper': 'paper',
    'Non-metallic minerals': 'non_metallic_minerals',
    'non_metallic_minerals': 'non_metallic_minerals',
    'non metallic minerals': 'non_metallic_minerals',
    'Other metals': 'other_metals',
    'other_metals': 'other_metals',
    'other metals': 'other_metals',
    'Machinery': 'machinery',
    'machinery': 'machinery',
    'Textiles and leather': 'textile_and_leather',
    'textile_and_leather': 'textile_and_leather',
    'Textile and leather': 'textile_and_leather',
    'Transport equipment': 'transport_equipment',
    'transport_equipment': 'transport_equipment',
    'Mining and quarrying': 'mining_and_quarrying',
    'mining_and_quarrying': 'mining_and_quarrying',
    'Central ICT': 'central_ict',
    'central_ict': 'central_ict',
    'Other': 'other',
    'other': 'other',
    'Gebruiker stuurt op via API': 'gebruiker_stuurt_op_via_api',
}

In [59]:
result = mapping.with_columns([
    pl.col("Cluster")
    .replace(cluster_map),

    pl.col('Sector').replace(sector_map)
])

In [60]:
result

Name,Name reformatted,Sector,Cluster,API input name,DSH plant name,DSH plant id,category,New site,Bottom-up
str,str,str,str,str,str,str,str,bool,bool
"""A12 CPP Petrogas EP Netherland…","""a12_cpp_petrogas_ep_netherland…","""other""","""cluster_6""","""other&&cluster_6&&a12_cpp_petr…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Aardgasbuffer Zuidwending""","""aardgasbuffer_zuidwending""","""other""","""cluster_6""","""other&&cluster_6&&aardgasbuffe…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Abbott Healthcare Products""","""abbott_healthcare_products""","""other_chemicals""","""cluster_6""","""other_chemicals&&cluster_6&&ab…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""Abbott Laboratories""","""abbott_laboratories""","""food""","""cluster_6""","""food&&cluster_6&&abbott_labora…",null,null,"""Bestaande niet-bottom-up sites""",false,false
"""ADM Europoort""","""adm_europoort""","""food""","""rotterdam_moerdijk""","""food&&rotterdam_moerdijk&&adm_…","""ADM Europoort""","""527be9a2-5258-4cda-89cb-b48520…","""Bestaande niet-bottom-up sites""",false,false
…,…,…,…,…,…,…,…,…,…
"""##new_cc_site230##""","""##new_cc_site230##""","""gebruiker_stuurt_op_via_api""","""gebruiker_stuurt_op_via_api""","""##new_cc_site230##""",null,null,"""New sites""",true,false
"""##new_cc_site231##""","""##new_cc_site231##""","""gebruiker_stuurt_op_via_api""","""gebruiker_stuurt_op_via_api""","""##new_cc_site231##""",null,null,"""New sites""",true,false
"""##new_cc_site232##""","""##new_cc_site232##""","""gebruiker_stuurt_op_via_api""","""gebruiker_stuurt_op_via_api""","""##new_cc_site232##""",null,null,"""New sites""",true,false
